# Linear Classification
An MNIST classification model using MLP: ReLU as the activation function, dropout regularization, and cross entropy as the loss function.

Architecture:
- flatten images into 1d tensors
- for all layers other than the output layer, perform LazyLinear, ReLU, and Dropout
- output 10 LazyLinear nodes

Hyperparameters:
- trained over 10 epochs
- data in batches of 64 images
- default learning rate is 0.03

In [1]:
# loading data

from torchvision import datasets, transforms
from torch import nn

from torch.utils.data import Dataset, DataLoader
from torch import optim
import torch

In [2]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform= transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [4]:
OUTPUT = 10
class LinearClassification(nn.Module):
    def __init__(self, *num_hidden, num_output= OUTPUT):
        super().__init__()
        nodes = [*num_hidden, num_output]
        layers = []
        
        layers.append(nn.Flatten())
        for i in range(len(nodes)):
            if i == len(nodes) - 1:
                layers.append(nn.LazyLinear(num_output))
            else:
                layers.extend([nn.LazyLinear(nodes[i]), nn.ReLU(), nn.Dropout(0.2)])

        self.net = nn.Sequential(*layers)
    
    def forward(self, X):
        return self.net(X)

In [5]:
# training time

def initialize_obj(*hiddens, lr):
    model = LinearClassification(*hiddens)
    optimizer = optim.SGD(model.parameters(), lr = lr)
    return model, optimizer

def train(model, optimizer):

    model.train()
    
    loss = nn.CrossEntropyLoss()

    for batch_img, batch_label in train_loader:
        y_hat = model.forward(batch_img)
        output = loss(y_hat, batch_label)
        optimizer.zero_grad()
        output.backward()
        optimizer.step()


In [6]:
# epoch loops
MAX_EPOCH = 10

model, optimizer = initialize_obj(20,19,18,17, lr = 0.03)

for _ in range(MAX_EPOCH):
    train(model, optimizer)

In [7]:

def eval(model):
    model.eval()
    batch_correct = []
    with torch.no_grad():
        for batch_img, batch_label in test_loader:
            correct = 0
            y_hat = model.forward(batch_img)
            predicted = torch.argmax(y_hat, dim=1)
            
            results  = predicted == batch_label
            correct = sum(results)
            batch_correct.append(correct/64)
    
    return sum(batch_correct)/len(batch_correct)
    

In [9]:
accuracy = eval(model)
print(f'Achieved {float(accuracy)} accuracy.')

Achieved 0.8957006335258484 accuracy.
